#Multi-Agent System with Shared Memory

#AI Research team

Pipeline

Orchestrator → Search → Analyst → Writer → QA

All agents read/write to shared memory

In [ ]:
!pip install groq

In [3]:
# SETUP
from groq import Groq
from google.colab import userdata
import json

api_key = userdata.get("newapi")

if not api_key:
    raise ValueError("API key not found")

client = Groq(api_key=api_key)
MODEL_NAME = "llama-3.3-70b-versatile"

In [4]:
# SHARED MEMORY (CORE CONCEPT)
class SharedMemory:
    def __init__(self):
        self.data = {}

    def update(self, agent_name, value):
        print(f"\n[Memory] Updating from {agent_name}")
        self.data[agent_name] = value

    def get(self, agent_name):
        return self.data.get(agent_name, {})

    def get_all(self):
        return self.data

In [5]:
# HELPERS
def clean_response(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.replace("```json", "").replace("```", "").strip()
    return text


def safe_parse_json(text):
    try:
        return json.loads(text)
    except:
        return {"error": "Invalid JSON", "raw_output": text}


def call_llm(prompt):
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Return ONLY valid JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )

        content = response.choices[0].message.content
        content = clean_response(content)

        return safe_parse_json(content)

    except Exception as e:
        return {"error": str(e)}

In [6]:
# Search Agent
class SearchAgent:
    name = "search"

    def run(self, goal, memory):
        print("\n[Search Agent] Fetching EV data...")

        prompt = f"""
        Collect key EV market trends in India for 2025.

        Return JSON:
        {{
          "trends": [],
          "statistics": []
        }}
        """

        return call_llm(prompt)

In [7]:
# Analyst Agent
class AnalystAgent:
    name = "analyst"

    def run(self, goal, memory):
        print("\n[Analyst Agent] Analyzing data...")

        search_data = memory.get("search")

        prompt = f"""
        Analyze this data:
        {json.dumps(search_data)}

        Return JSON:
        {{
          "insights": [],
          "summary": ""
        }}
        """

        return call_llm(prompt)

In [8]:
# Writer Agent
class WriterAgent:
    name = "writer"

    def run(self, goal, memory):
        print("\n[Writer Agent] Writing report...")

        insights = memory.get("analyst")

        prompt = f"""
        Write a professional EV market report for India 2025.

        Based on:
        {json.dumps(insights)}

        Return JSON:
        {{
          "report": ""
        }}
        """

        return call_llm(prompt)


In [9]:
# QA agent
class QAAgent:
    name = "qa"

    def run(self, goal, memory):
        print("\n[QA Agent] Reviewing report...")

        report = memory.get("writer")

        prompt = f"""
        Review this report:
        {json.dumps(report)}

        Improve clarity, correctness, tone.

        Return JSON:
        {{
          "final_report": ""
        }}
        """

        return call_llm(prompt)

In [10]:
# Orchestrator
def orchestrator(goal):
    print("Goal:", goal)

    agents = [
        SearchAgent(),
        AnalystAgent(),
        WriterAgent(),
        QAAgent()
    ]

    memory = SharedMemory()

    for agent in agents:
        result = agent.run(goal, memory)
        memory.update(agent.name, result)

    return memory.get_all()

In [11]:
# RUN
goal = "Write a comprehensive market report on EV industry trends in India for 2025"

output = orchestrator(goal)

print("\n===== FINAL OUTPUT =====")
print(json.dumps(output, indent=2))

Goal: Write a comprehensive market report on EV industry trends in India for 2025

[Search Agent] Fetching EV data...

[Memory] Updating from search

[Analyst Agent] Analyzing data...

[Memory] Updating from analyst

[Writer Agent] Writing report...

[Memory] Updating from writer

[QA Agent] Reviewing report...

[Memory] Updating from qa

===== FINAL OUTPUT =====
{
  "search": {
    "trends": [
      "Increasing Demand for Electric Two-Wheelers",
      "Growing Adoption of Electric Cars",
      "Expansion of Charging Infrastructure",
      "Government Incentives and Subsidies",
      "Rising Investment in EV Manufacturing",
      "Development of New Business Models",
      "Improving Battery Technology",
      "Growing Focus on Sustainable Mobility"
    ],
    "statistics": [
      {
        "category": "Electric Two-Wheelers",
        "value": "Expected to reach 5 million units sold by 2025"
      },
      {
        "category": "Electric Cars",
        "value": "Projected to capture 1